# OptoJump Annotation Review
Inspect saved landing / takeoff annotations for a single athlete test.
Set the parameters in **Cell 1**, then run all cells (`Kernel → Restart & Run All`).

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
ATHLETE_NAME = "Dawid Bucki"
STUDY_ID = 1
TEST_ID = 1

ANNOTATIONS_CSV = "../data/output/annotations/optojump/ml_training_dataset.csv"
VIDEO_DIR = "../data/input/optojump/"
RECORDING_FPS = 120
VIDEO_FPS = 30

# How many steps to show in the frame-strip panel (None = all)
STEPS_TO_SHOW = None

In [ ]:
import os, glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets as widgets

plt.rcParams.update(
    {
        "figure.facecolor": "#0f1117",
        "axes.facecolor": "#1a1d27",
        "axes.edgecolor": "#3a3d4d",
        "text.color": "#e0e0e0",
        "axes.labelcolor": "#e0e0e0",
        "xtick.color": "#a0a0b0",
        "ytick.color": "#a0a0b0",
        "grid.color": "#2a2d3a",
        "grid.linewidth": 0.6,
        "font.family": "monospace",
    }
)

GREEN = "#00e676"
RED = "#ff5252"
YELLOW = "#ffd740"
DIM = "#3a3d4d"

In [ ]:
def extract_frame(cap: cv2.VideoCapture, frame_idx: int):
    """Seek `cap` to `frame_idx` and return an RGB numpy array, or None on failure."""
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    success, frame = cap.read()
    if not success:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def overlay_label(img, label: str, color: tuple) -> np.ndarray:
    """Return a copy of `img` with `label` text drawn in `color` (BGR tuple)."""
    out = img.copy()
    cv2.putText(
        out, label,
        org=(8, 28),
        fontFace=cv2.FONT_HERSHEY_SIMPLEX,
        fontScale=0.7,
        color=color,
        thickness=2,
        lineType=cv2.LINE_AA,
    )
    return out

## 1 · Load annotations

In [ ]:
raw = pd.read_csv(ANNOTATIONS_CSV)

# ── resolve which video file belongs to this athlete / study / test ────────
pattern = os.path.join(
    VIDEO_DIR,
    f"study_{STUDY_ID}",
    f"{ATHLETE_NAME.lower().replace(' ', '_')}_{TEST_ID}.mov",
)
matches = glob.glob(pattern)
VIDEO_PATH = matches[0] if matches else None
if VIDEO_PATH is None:
    print(f"⚠  Video not found at: {pattern}")

video_filename = f"{ATHLETE_NAME.lower().replace(' ', '_')}_{TEST_ID}.mov"
video_path_suffix = os.path.join(f"study_{STUDY_ID}", video_filename)

label_df = (
    raw[raw["video_path"].str.endswith(video_path_suffix)].copy()
    if not raw.empty
    else pd.DataFrame()
)

# ── rebuild event-level annotations from contact/flight boundaries ─────────
# A "touchdown" is the first 'contact' frame after a 'flight' block;
# a "takeoff" is the first 'flight' frame after a 'contact' block.
if not label_df.empty:
    label_df = label_df.sort_values("frame_number").reset_index(drop=True)
    shifted = label_df["label"].shift(1, fill_value="flight")
    td_mask = (label_df["label"] == "contact") & (shifted == "flight")
    to_mask = (label_df["label"] == "flight") & (shifted == "contact")

    td_frames = label_df.loc[td_mask, "frame_number"].tolist()
    to_frames = label_df.loc[to_mask, "frame_number"].tolist()

    event_rows = []
    for i, (td, to) in enumerate(zip(td_frames, to_frames), start=1):
        td_sec = round(td / RECORDING_FPS, 3)
        to_sec = round(to / RECORDING_FPS, 3)
        ct_ms = round((to - td) / RECORDING_FPS * 1000)
        event_rows.extend(
            [
                {
                    "step": i,
                    "event": "touchdown",
                    "frame": td,
                    "time_sec": td_sec,
                    "contact_ms": ct_ms,
                },
                {
                    "step": i,
                    "event": "takeoff",
                    "frame": to,
                    "time_sec": to_sec,
                    "contact_ms": ct_ms,
                },
            ]
        )
    event_df = pd.DataFrame(event_rows)
else:
    event_df = pd.DataFrame()

n_steps = event_df["step"].nunique() if not event_df.empty else 0
n_frames = len(label_df)
n_contact = (label_df["label"] == "contact").sum() if not label_df.empty else 0
n_flight = (label_df["label"] == "flight").sum() if not label_df.empty else 0

print(f"Athlete : {ATHLETE_NAME}   |   Study {STUDY_ID}  ·  Test {TEST_ID}")
print(f"Video   : {video_path_suffix}")
print(f"Frames  : {n_frames:,}  ({n_contact:,} contact  /  {n_flight:,} flight)")
print(f"Steps   : {n_steps}")

## 2 · Per-step summary table

In [ ]:
if event_df.empty:
    print("No annotations found.")
else:
    td = event_df[event_df["event"] == "touchdown"].set_index("step")
    to = event_df[event_df["event"] == "takeoff"].set_index("step")

    summary = pd.DataFrame(
        {
            "touchdown_frame": td["frame"],
            "touchdown_sec": td["time_sec"],
            "takeoff_frame": to["frame"],
            "takeoff_sec": to["time_sec"],
            "contact_ms": td["contact_ms"],
        }
    ).rename_axis("step")

    display(
        summary.style.format(
            {
                "touchdown_sec": "{:.3f}",
                "takeoff_sec": "{:.3f}",
                "contact_ms": "{:d} ms",
            }
        )
        .background_gradient(subset=["contact_ms"], cmap="YlOrRd")
        .set_caption(f"{ATHLETE_NAME}  —  Study {STUDY_ID}  ·  Test {TEST_ID}")
    )

## 3 · Interactive step scrubber
Use the slider to jump to any step and inspect the surrounding frames in detail.
Requires `ipywidgets` and a running Jupyter kernel.

In [ ]:
def show_step(step_num: int, context: int = 1) -> None:
    """Display context frames around touchdown and takeoff for one step."""
    if event_df.empty or VIDEO_PATH is None:
        print("No data.")
        return

    step_events = event_df[event_df["step"] == step_num]
    td_row = step_events[step_events["event"] == "touchdown"]
    to_row = step_events[step_events["event"] == "takeoff"]
    if td_row.empty or to_row.empty:
        print(f"Step {step_num} incomplete.")
        return

    td_f = int(td_row.iloc[0]["frame"])
    to_f = int(to_row.iloc[0]["frame"])
    ct_ms = int(td_row.iloc[0]["contact_ms"])

    cap = cv2.VideoCapture(VIDEO_PATH)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    td_range = [max(0, td_f + i) for i in range(-context, context + 1)]
    to_range = [min(total - 1, to_f + i) for i in range(-context, context + 1)]
    n_cols = len(td_range)

    fig, axes = plt.subplots(2, n_cols, figsize=(10 * n_cols, 15))
    fig.patch.set_facecolor("#0f1117")
    fig.suptitle(
        f"Step {step_num}   |   touchdown f{td_f}  →  takeoff f{to_f}   |   contact {ct_ms} ms",
        color=YELLOW,
        fontsize=11,
        y=1.01,
    )

    for col, (td_fi, to_fi) in enumerate(zip(td_range, to_range)):
        offset = td_fi - td_f
        for row_ax, fidx, base_color, event_label in (
            (0, td_fi, GREEN, "TOUCHDOWN"),
            (1, to_fi, RED, "TAKEOFF"),
        ):
            ax = axes[row_ax][col]
            img = extract_frame(cap, fidx)
            if img is not None:
                color = (0, 230, 118) if base_color == GREEN else (255, 82, 82)
                label = f"{'+' if offset >= 0 else ''}{offset}  f{fidx}"
                ax.imshow(overlay_label(img, label, color))
            ax.axis("off")

            if offset == 0:
                ax.set_title(
                    event_label, color=base_color, fontsize=8, pad=3, fontweight="bold"
                )
                for spine in ax.spines.values():
                    spine.set_visible(True)
                    spine.set_edgecolor(base_color)
                    spine.set_linewidth(2.5)

    cap.release()
    plt.tight_layout()
    plt.show()


if not event_df.empty and n_steps > 0:
    step_slider = widgets.IntSlider(
        value=1,
        min=1,
        max=n_steps,
        description="Step:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="400px"),
    )
    ctx_slider = widgets.IntSlider(
        value=1,
        min=1,
        max=8,
        description="Context frames:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="400px"),
    )
    ui = widgets.VBox([step_slider, ctx_slider])
    out = widgets.interactive_output(
        show_step, {"step_num": step_slider, "context": ctx_slider}
    )
    display(ui, out)
else:
    print("No steps to display.")

## 4 · Pose kinematics

Load the corresponding YOLO pose data and plot y-position, y-velocity, and y-acceleration for a selected keypoint.
Landing (touchdown) and takeoff moments are marked from the annotations loaded in Cell 1.

In [ ]:
import json

from src.gait_analysis.data_cleaning.data_smoothing import smooth_pose_data

# cv2 reports 30 fps for these videos even though the actual recording rate is
# RECORDING_FPS = 120.  The YOLO processor uses cap.get(cv2.CAP_PROP_FPS) to
# compute timestamp_ms, so every stored timestamp is (120/30) = 4× too large.
# However, cv2 still reads *every* frame, so the data is truly sampled at
# RECORDING_FPS and must be smoothed at that rate.
CV2_FPS = 30

# ── resolve pose file path ─────────────────────────────────────────────────
athlete_key = ATHLETE_NAME.lower().replace(" ", "_")
pose_path = os.path.join(
    "../data/output/tuned_yolo/000332/optojump",
    f"study_{STUDY_ID}",
    f"{athlete_key}_{TEST_ID}.json",
)

if not os.path.exists(pose_path):
    print(f"⚠  Pose file not found: {pose_path}")
    smooth_data_opt = None
else:
    with open(pose_path) as f:
        pose_json = json.load(f)

    connections_opt = pose_json["connections"]
    pose_data_raw = pose_json["pose_data"]

    keypoints_opt = list(set(kp for pair in connections_opt for kp in pair))
    anchors_opt = [kp.replace('right_', '').replace('left_', '') for kp in keypoints_opt if any(a in kp for a in ["heel", "big_toe", "knee"])]

    smooth_data_opt = smooth_pose_data(
        pose_data=pose_data_raw,
        keypoints=keypoints_opt,
        anchors=anchors_opt,
        keys_to_exclude={"raw_keypoints"},
        fps=RECORDING_FPS,    # actual sample rate (cv2 reads every frame)
        cutoff=4.0,
    )

    # Correct timestamps to real wall-clock time: the processor divided by
    # cv2's reported fps (30) instead of the actual fps (120), so timestamps
    # are 4× too large.
    smooth_data_opt["timestamp_ms"] *= CV2_FPS / RECORDING_FPS

    smooth_data_opt["frame"] = (
        smooth_data_opt["timestamp_ms"] / 1000 * RECORDING_FPS
    ).astype(int)
    smooth_data_opt = smooth_data_opt.set_index("frame")

    available_parts = sorted(set(
        col[len(side) + 1 : col.rfind("_")]
        for col in smooth_data_opt.columns
        if col.endswith("_y")
        for side in ("left", "right")
        if col.startswith(side + "_")
    ))

    print(f"Loaded: {pose_path}")
    print(f"Frames : {len(smooth_data_opt)}")
    print(f"Time   : {smooth_data_opt['timestamp_ms'].min():.0f} – {smooth_data_opt['timestamp_ms'].max():.0f} ms")
    print(f"Parts  : {available_parts}")

In [ ]:
# ── choose your keypoint ──────────────────────────────────────────────────
SIDE = "left"      # "right" or "left"
KEYPOINT = "ankle"  # e.g. "ankle", "heel", "big_toe", "knee", "hip", "shoulder"

# ─────────────────────────────────────────────────────────────────────────
from src.gait_analysis.parameter_calculation.gait_events.gait_events_detection import (
    detect_landings,
    detect_liftoffs,
    enforce_alternating,
)

if smooth_data_opt is None:
    print("Run the pose-loading cell above first.")
else:
    col = f"{SIDE}_{KEYPOINT}_y"
    if col not in smooth_data_opt.columns:
        print(f"Column '{col}' not found.\nAvailable parts: {available_parts}")
    else:
        dt = 1 / RECORDING_FPS
        pos = smooth_data_opt[col].values * -1
        vel = np.gradient(pos, dt)
        acc = np.gradient(vel, dt)

        def _norm(arr):
            lo, hi = arr.min(), arr.max()
            return (arr - lo) / (hi - lo) if hi > lo else np.zeros_like(arr)

        time_x = smooth_data_opt["timestamp_ms"].values
        pos_n, vel_n, acc_n = _norm(pos), _norm(vel), _norm(acc)

        # ── detect events and enforce alternating order ────────────────────
        landing_times = detect_landings(vel, time_x)
        liftoff_times = detect_liftoffs(vel, time_x)
        landing_times, liftoff_times = enforce_alternating(landing_times, liftoff_times)

        # ── ground-truth events from CSV (converted to ms) ────────────────
        gt_td_times = event_df.loc[event_df["event"] == "touchdown", "frame"].values / RECORDING_FPS * 1000
        gt_to_times = event_df.loc[event_df["event"] == "takeoff",   "frame"].values / RECORDING_FPS * 1000

        def _plot_signals(ax):
            ax.plot(time_x, pos_n, label="Position (y)",     color="#1f77b4", lw=2.5)
            ax.plot(time_x, vel_n, label="Velocity (y)",     color="#ff7f0e", ls="--", lw=1.5)
            ax.plot(time_x, acc_n, label="Acceleration (y)", color="#2ca02c", ls=":",  lw=1.5)
            ax.set_xlabel("Time (ms)", fontsize=11)
            ax.set_ylabel("Normalised magnitude", fontsize=11)
            ax.set_ylim(-0.05, 1.10)
            ax.grid(True, alpha=0.15)

        def _mark_events(ax, td_times, to_times, td_label, to_label):
            for i, t in enumerate(td_times):
                ax.axvline(x=t, color=GREEN, alpha=0.8, lw=1.2,
                           label=td_label if i == 0 else None)
                ax.scatter(t, 1.06, color=GREEN, marker="v", s=100, zorder=5, clip_on=False)
            for i, t in enumerate(to_times):
                ax.axvline(x=t, color=RED, alpha=0.8, lw=1.2,
                           label=to_label if i == 0 else None)
                ax.scatter(t, 1.06, color=RED, marker="^", s=100, zorder=5, clip_on=False)

        title_base = (
            f"{SIDE.capitalize()} {KEYPOINT.capitalize()}  —  "
            f"{ATHLETE_NAME}, Study {STUDY_ID}, Test {TEST_ID}"
        )

        # ── figure 1: ground truth ────────────────────────────────────────
        fig1, ax1 = plt.subplots(figsize=(15, 5))
        _plot_signals(ax1)
        _mark_events(ax1, gt_td_times, gt_to_times,
                     td_label="Landing — ground truth (CSV)",
                     to_label="Takeoff — ground truth (CSV)")
        ax1.set_title(f"{title_base}\nGround truth (OptoJump CSV)", fontsize=12, pad=30)
        ax1.legend(loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0, framealpha=0.8)
        plt.tight_layout()
        plt.show()

        # ── figure 2: kinematic detection ─────────────────────────────────
        fig2, ax2 = plt.subplots(figsize=(15, 5))
        _plot_signals(ax2)
        _mark_events(ax2, landing_times, liftoff_times,
                     td_label="Landing — vel = 0",
                     to_label="Liftoff — max velocity")
        ax2.set_title(f"{title_base}\nKinematic detection", fontsize=12, pad=30)
        ax2.legend(loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0, framealpha=0.8)
        plt.tight_layout()
        plt.show()